# 노이즈 필터 개발

센서 원신호에 EMA(Exponential Moving Average) 필터를 적용하여
노이즈를 억제하면서도 실제 상태 변화에 빠르게 반응하는 최적 파라미터를 탐색한다.

확정된 상수(`ALPHA`)는 `backend/pipeline/noise_filter.py`에 이식된다.

## 1. 데이터 생성

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

SAMPLE_RATE = 50   # Hz
DURATION    = 10   # 초
N = SAMPLE_RATE * DURATION
t = np.linspace(0, DURATION, N)

# 실제 신호: 이동 중(0~4s) → 거치됨(4~10s)
true_accel = np.where(t < 4, 1.0 + 0.3 * np.sin(2 * np.pi * 2 * t), 1.0)
true_mag_x = np.where(t < 4, 20.0 + 5 * np.sin(2 * np.pi * 0.5 * t), 30.5)

# 가우시안 노이즈 주입
NOISE_STD = 0.05
noisy_accel = true_accel + np.random.normal(0, NOISE_STD, N)
noisy_mag_x = true_mag_x + np.random.normal(0, NOISE_STD * 3, N)

fig, axes = plt.subplots(2, 1, figsize=(12, 5))
axes[0].plot(t, true_accel, label='true', alpha=0.5)
axes[0].plot(t, noisy_accel, label='noisy', alpha=0.7)
axes[0].set_title('accel_magnitude')
axes[0].legend()
axes[1].plot(t, true_mag_x, label='true', alpha=0.5)
axes[1].plot(t, noisy_mag_x, label='noisy', alpha=0.7)
axes[1].set_title('mag_x')
axes[1].legend()
plt.tight_layout()
plt.show()

## 2. EMA 구현

In [ ]:
def apply_ema(signal: np.ndarray, alpha: float) -> np.ndarray:
    out = np.empty_like(signal)
    out[0] = signal[0]
    for i in range(1, len(signal)):
        out[i] = alpha * signal[i] + (1 - alpha) * out[i - 1]
    return out

alphas = [0.1, 0.3, 0.5, 0.7, 0.9]
plt.figure(figsize=(12, 4))
plt.plot(t, true_accel, 'k--', label='true', alpha=0.4)
plt.plot(t, noisy_accel, color='gray', label='noisy', alpha=0.3)
for a in alphas:
    plt.plot(t, apply_ema(noisy_accel, a), label=f'alpha={a}')
plt.title('EMA alpha 비교 — accel_magnitude')
plt.legend()
plt.tight_layout()
plt.show()

## 4. 상수 확정

그리드 탐색 결과:
- accel 최적 alpha = **0.65**  RMSE = 0.0379
- mag   최적 alpha = **0.95**  RMSE = 0.1419

두 채널에 단일 alpha를 적용하므로 accel(imu_state 주 신호) 기준 **0.65** 채택.
mag 채널은 MIN_SETTLE_SAMPLES 평균으로 노이즈를 별도 억제하므로 0.65로 충분.

`ALPHA = 0.65`를 `backend/pipeline/noise_filter.py`에 적용.

In [ ]:
alphas_grid = np.arange(0.05, 1.0, 0.05)
rmse_accel = []
rmse_mag   = []

for a in alphas_grid:
    filtered_a = apply_ema(noisy_accel, a)
    filtered_m = apply_ema(noisy_mag_x, a)
    rmse_accel.append(np.sqrt(np.mean((filtered_a - true_accel) ** 2)))
    rmse_mag.append(np.sqrt(np.mean((filtered_m - true_mag_x) ** 2)))

best_alpha_accel = alphas_grid[np.argmin(rmse_accel)]
best_alpha_mag   = alphas_grid[np.argmin(rmse_mag)]
print(f'accel best alpha: {best_alpha_accel:.2f}  RMSE={min(rmse_accel):.4f}')
print(f'mag   best alpha: {best_alpha_mag:.2f}  RMSE={min(rmse_mag):.4f}')

plt.figure(figsize=(8, 4))
plt.plot(alphas_grid, rmse_accel, label='accel RMSE')
plt.plot(alphas_grid, rmse_mag, label='mag RMSE')
plt.axvline(0.65, color='red', linestyle='--', label='adopted (0.65)')
plt.xlabel('alpha')
plt.ylabel('RMSE')
plt.title('alpha vs RMSE')
plt.legend()
plt.tight_layout()
plt.show()

## 4. 상수 확정

그리드 탐색 결과:
- accel 최적 alpha = **0.65**  RMSE = 0.0379
- mag   최적 alpha = **0.95**  RMSE = 0.1419

두 채널에 단일 alpha를 적용하므로 accel(imu_state 주 신호) 기준 **0.65** 채택.
mag 채널은 MIN_SETTLE_SAMPLES 평균으로 노이즈를 별도 억제하므로 0.65로 충분.

`ALPHA = 0.65`를 `backend/pipeline/noise_filter.py`에 적용.

In [ ]:
ALPHA = 0.65
print(f'noise_filter.py 에 적용할 상수: ALPHA = {ALPHA}')